# 101 --- Getting Started with ThetaDataDx

This notebook walks through the basics of the `thetadatadx` Python SDK:
installing the package, authenticating, fetching stock and option data,
and computing option Greeks --- all powered by a native Rust backend.

**Prerequisites:** A ThetaData account with an active subscription.

## 1. Installation

Install the SDK with pandas support. The package ships a pre-compiled
Rust binary via PyO3 --- no JVM required.

In [1]:
# Uncomment to install (run once)
# !pip install thetadatadx[pandas]

## 2. Connect with Credentials

Create a `Credentials` object with your ThetaData email and password,
then connect a `ThetaDataDx` client to the production servers.

In [2]:
from thetadatadx import Credentials, Config, ThetaDataDx, to_dataframe

# Option A: inline credentials
creds = Credentials.from_file("creds.txt")

# Option B: load from a file (line 1 = email, line 2 = password)
creds = Credentials.from_file("creds.txt")

config = Config.production()
tdx = ThetaDataDx(creds, config)
print("Connected!", tdx)

Connected! ThetaDataDx(historical=connected, streaming=none)


## 3. List Stock Symbols

Retrieve the full universe of available stock tickers.

In [3]:
symbols = tdx.stock_list_symbols()
print(f"Total symbols available: {len(symbols)}")
print(f"First 20: {symbols[:20]}")

Total symbols available: 25472
First 20: ['.PR.I.PR.A', '.PR.I.PR.B', '.PR.S/WI', 'A', 'AA', 'AAA', 'AAAA', 'AAAC', 'AAAP', 'AAAU', 'AABA', 'AAC', 'AAC.U', 'AAC.WS', 'AACB', 'AACBR', 'AACBU', 'AACC', 'AACG', 'AACI']


## 4. Fetch AAPL End-of-Day Data as a DataFrame

`stock_history_eod` returns a list of tick dicts. Convert to a pandas
DataFrame with `to_dataframe()`.

In [4]:
import pandas as pd

eod_ticks = tdx.stock_history_eod("AAPL", "20240101", "20240401")
df_eod = to_dataframe(eod_ticks)

print(f"Rows: {len(df_eod)}")
df_eod.head(10)

Rows: 62


,ms_of_day,ms_of_day2,open,high,low,close,volume,count,bid_size,bid_exchange,bid,bid_condition,ask_size,ask_exchange,ask,ask_condition,date,expiration,strike,right
0,62273606,62271877,187.03,188.4400,183.885,185.64,80680243,1003582,2,7,185.34,0,2,1,185.36,0,20240102,0,0.0,
1,62189883,62188586,184.20,185.8800,183.430,184.25,58308345,654127,5,7,184.05,0,2,1,184.10,0,20240103,0,0.0,
2,62226020,62222445,182.00,183.0872,180.880,181.91,71197269,709246,8,60,181.76,0,2,7,181.79,0,20240104,0,0.0,
3,62217032,62209821,181.90,182.7600,180.170,181.18,61949135,679405,3,1,181.03,0,3,7,181.05,0,20240105,0,0.0,
4,62221830,62216974,182.00,185.6000,181.500,185.56,59028985,665625,4,1,185.28,0,1,65,185.37,0,20240108,0,0.0,
5,62224950,62217318,183.96,185.1500,182.730,185.14,42783541,535940,1,1,185.22,0,11,65,185.24,0,20240109,0,0.0,
6,62117022,62110004,184.28,186.4000,183.920,186.19,46046017,551340,3,1,186.68,0,3,1,186.72,0,20240110,0,0.0,
7,62171475,62171317,186.60,187.0500,183.620,185.59,49016579,581769,2,7,185.63,0,3,1,185.66,0,20240111,0,0.0,
8,62102041,62095959,186.00,186.7400,185.190,185.92,40246347,475237,1,1,185.84,0,1,7,185.87,0,20240112,0,0.0,
9,62128582,62125236,182.23,184.2600,180.934,183.63,65482123,764555,4,1,183.37,0,1,65,183.44,0,20240116,0,0.0,


## 5. Fetch 1-Minute OHLC Bars

Intraday bars for a single trading day. The `interval` argument is
the bar width in milliseconds: `"60000"` = 1 minute.

In [5]:
ohlc_ticks = tdx.stock_history_ohlc("AAPL", "20240315", "60000")
df_ohlc = to_dataframe(ohlc_ticks)

print(f"1-min bars on 2024-03-15: {len(df_ohlc)}")
df_ohlc.head(10)

1-min bars on 2024-03-15: 391


,ms_of_day,open,high,low,close,volume,count,date,expiration,strike,right
0,34200000,171.170,172.4800,171.00,171.7100,12180669,17420,20240315,0,0.0,
1,34260000,171.690,171.9000,171.43,171.5874,370342,5041,20240315,0,0.0,
2,34320000,171.580,171.8261,171.35,171.4400,373855,4935,20240315,0,0.0,
3,34380000,171.435,171.6800,171.33,171.4100,320913,4456,20240315,0,0.0,
4,34440000,171.400,171.9000,171.39,171.7450,374921,4726,20240315,0,0.0,
5,34500000,171.738,171.8900,171.67,171.8200,300985,4342,20240315,0,0.0,
6,34560000,171.820,171.8700,171.68,171.7850,306493,4006,20240315,0,0.0,
7,34620000,171.780,171.8100,171.67,171.7130,1805443,4409,20240315,0,0.0,
8,34680000,171.710,171.7300,171.41,171.5400,326681,3720,20240315,0,0.0,
9,34740000,171.550,171.5989,171.43,171.4300,1327948,2922,20240315,0,0.0,


## 6. List SPY Option Expirations and Strikes

Options data starts with listing available expirations, then
the strikes for a given expiration.

In [6]:
# All available expirations for SPY
expirations = tdx.option_list_expirations("SPY")
print(f"Total expirations: {len(expirations)}")
print(f"Next 5: {expirations[:5]}")

# Strikes for the nearest expiration
nearest_exp = expirations[0]
strikes = tdx.option_list_strikes("SPY", nearest_exp)
print(f"\nStrikes for {nearest_exp}: {len(strikes)} total")
print(f"Range: {strikes[0]} -- {strikes[-1]}")
print(f"Sample: {strikes[len(strikes)//2 - 3 : len(strikes)//2 + 3]}")

Total expirations: 2025
Next 5: ['2012-06-01', '2012-06-08', '2012-06-16', '2012-06-22', '2012-06-29']



Strikes for 2012-06-01: 27 total
Range: 120 -- 139
Sample: ['119', '127', '135', '116', '124', '132']


## 7. Greeks Calculator

The SDK includes a built-in Black-Scholes Greeks calculator written
in Rust. No external dependencies needed --- just pass the inputs.

- `all_greeks()` returns IV + all Greeks (first, second, third order)
- `implied_volatility()` returns just the IV and the numerical error

In [7]:
from thetadatadx import all_greeks, implied_volatility

# SPY ATM call: spot=$540, strike=$540, r=5%, div yield=1.3%, 30 DTE, mid=$9.80
greeks = all_greeks(
    spot=540.0,
    strike=540.0,
    rate=0.05,
    div_yield=0.013,
    tte=30.0 / 365.0,
    option_price=9.80,
    right="C",
)

print("=== SPY 540C  30 DTE ===")
print(f"  IV:        {greeks['iv']:.4f}  (error: {greeks['iv_error']:.2e})")
print(f"  Delta:     {greeks['delta']:.4f}")
print(f"  Gamma:     {greeks['gamma']:.6f}")
print(f"  Theta:     {greeks['theta']:.4f}")
print(f"  Vega:      {greeks['vega']:.4f}")
print(f"  Rho:       {greeks['rho']:.4f}")
print(f"  Vanna:     {greeks['vanna']:.6f}")
print(f"  Charm:     {greeks['charm']:.6f}")
print(f"  Vomma:     {greeks['vomma']:.6f}")
print(f"  Speed:     {greeks['speed']:.8f}")
print(f"  Zomma:     {greeks['zomma']:.8f}")
print(f"  Color:     {greeks['color']:.8f}")
print(f"  Ultima:    {greeks['ultima']:.6f}")

=== SPY 540C  30 DTE ===
  IV:        0.1454  (error: 5.78e-06)
  Delta:     0.5368
  Gamma:     0.017624
  Theta:     -0.1769
  Vega:      61.4248
  Rho:       23.0191
  Vanna:     -0.142159
  Charm:     -0.219402
  Vomma:     2.064154
  Speed:     -0.00010606
  Zomma:     -0.12060755
  Color:     -0.10838750
  Ultima:    -47.563633


In [8]:
# Standalone implied volatility (faster if you only need IV)
iv, iv_err = implied_volatility(
    spot=540.0,
    strike=540.0,
    rate=0.05,
    div_yield=0.013,
    tte=30.0 / 365.0,
    option_price=9.80,
    right="C",
)
print(f"IV = {iv:.4f}  (numerical error: {iv_err:.2e})")

IV = 0.1454  (numerical error: 5.78e-06)


---

**Next:** [102 --- Historical Data Analysis](102_historical_analysis.ipynb)